# deepEmulator — DDQN with a frozen DINO encoder

Trains DDQN on Pokemon Red where the agent's observation is a **DINO latent** instead of a raw pixel stack. Reads the encoder bundle written by `notebook 04` (or local pretraining), wraps the env with `FrozenEncoderEnv`, and the agent learns on `(frame_stack * latent_dim,)` vectors via the MLP head of the generalized DDQN.

Re-running this cell after a Colab disconnect picks up training from the latest checkpoint under `MyDrive/deepEmulator/checkpoints/pokemon_red_dino/`.

**Prereqs in Drive:**
- `MyDrive/deepEmulator/encoders/dino/latest.txt` (or set `encoder` arg explicitly)
- `MyDrive/deepEmulator/roms/PokemonRed.gb`
- `MyDrive/deepEmulator/states/init.state`

In [ ]:
!nvidia-smi -L || echo 'no GPU'

In [ ]:
!pip install -q git+https://github.com/juangarassino/deepEmulator.git

In [ ]:
# Resolve latest DINO encoder bundle
from pathlib import Path
from google.colab import drive; drive.mount('/content/drive', force_remount=False)
encoder_root = Path('/content/drive/MyDrive/deepEmulator/encoders/dino')
latest_marker = encoder_root / 'latest.txt'
assert latest_marker.exists(), f'no latest.txt at {encoder_root} — run notebook 04 first'
encoder_path = Path(latest_marker.read_text().strip())
print('encoder:', encoder_path)

In [ ]:
# Train. Bundle goes to MyDrive/deepEmulator/checkpoints/pokemon_red/<run>/
# (metadata.json will record `encoder` pointing back at the DINO bundle.)
from deepEmulator.training.colab_train import run_in_colab

run_in_colab(
    cartridge='POKEMON RED',
    rom='deepEmulator/roms/PokemonRed.gb',
    init_state='deepEmulator/states/init.state',
    steps=100_000,
    save_every=10_000,
    encoder=str(encoder_path),
    resume=True,
)